# 07 — VAERS v11b: official MulTaBench run

**What this does:** runs the official MulTaBench `benchmark.py`, unchanged, at commit
[`c955466`](https://github.com/alanarazi7/MulTaBench/tree/c95546638be9bac472a2ee38afa7171060ab80bc)
(`master` as it stood on 26 Jul 2026 — its last commit is from 25 Jul 2026, 23:46 Israel time),
on the frozen v11b dataset: **5 learners × 4 conditions × 5 folds = 100 runs**.

The only addition to the official repo is registering the dataset (one enum line, one file-path line,
one curation recipe, the CSV). No model, split, metric or encoder code is touched — see
[`results/official_c955466/registration.diff`](../../results/official_c955466/registration.diff).

**This copy keeps the real outputs** of the run (Colab, NVIDIA A100-SXM4-80GB). The run cell is resumable:
the first 17 runs finished in an earlier session that Colab disconnected, so its output starts at run 18.
All 100 scores are in [`results/official_c955466/results.csv`](../../results/official_c955466/results.csv).

**How to re-run:** Colab → Runtime → Change runtime type → **L4 or A100** (not T4: the `ft` runs need the memory).
Put `vaers_v11b.csv` on Drive, set `CSV_PATH` / `OUT_DIR` in the settings cell, add the Colab secrets
`WANDB_API_KEY`, `WANDB_ENTITY`, `HF_TOKEN` (optional `TABPFN_TOKEN`), then Runtime → **Run all**.
Every finished run is appended to `results.csv` on Drive, so after a disconnect just **Run all** again.

In [ ]:
# ===== Settings (normally nothing to change) =====
COMMIT      = "c95546638be9bac472a2ee38afa7171060ab80bc"   # master at project opening (26 Jul 2026 00:00 IDT)
REPO_URL    = "https://github.com/alanarazi7/MulTaBench.git"
CSV_PATH    = "/content/drive/MyDrive/text-tabular-research-2/candidates/vaers_v11b/vaers_v11b.csv"
CSV_SHA256  = "8e59f09b19d0f7f174b38413ffe48052b005491fa1464877764d405911e16690"
DATASET     = "BIN_TEXT_VAERS_SERIOUS_V11B"
OUT_DIR     = "/content/drive/MyDrive/text-tabular-research-2/official-final/vaers_v11b_c955466"
WANDB_PROJECT = "multabench_vaers_final"
MODELS = ["light", "cat", "tabm", "tabpfnv2", "tabpfnv2p5"]
STATES = ["no_text", "text_only", "all", "ft"]   # Structured, Text, Joint Frozen, Joint TAR
FOLDS  = [0, 1, 2, 3, 4]
REPO   = "/content/MulTaBench"

In [ ]:
# the fine-tuned (ft) runs require CUDA_VISIBLE_DEVICES to be set
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
# ===== 1. Drive, GPU, official code, secrets =====
import os, subprocess, sys
from google.colab import drive, userdata
drive.mount("/content/drive")
os.makedirs(f"{OUT_DIR}/logs", exist_ok=True)

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "NONE")
assert gpu, "No GPU. Runtime -> Change runtime type -> L4"
assert "T4" not in gpu, "T4 is too small for the fine-tuned (ft) runs. Switch to L4 or A100."

if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
subprocess.run(["git", "-C", REPO, "checkout", "-q", "-f", COMMIT], check=True)
head = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
assert head == COMMIT, f"wrong commit {head}"
print("MulTaBench at", head[:7], "(release-day master)")

# secrets -> .env and environment (values are never printed)
needed = ["WANDB_API_KEY", "WANDB_ENTITY", "HF_TOKEN"]
optional = ["TABPFN_TOKEN"]
SECRETS = {}
for k in needed + optional:
    try:
        v = userdata.get(k)
    except Exception:
        v = None
    if not v and k in needed:
        raise RuntimeError(f"Colab secret {k} is missing or has no notebook access")
    if v:
        SECRETS[k] = v
with open(f"{REPO}/.env", "w") as f:
    for k, v in SECRETS.items():
        f.write(f"{k}={v}\n")
print("secrets loaded:", sorted(SECRETS))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: NVIDIA A100-SXM4-80GB, 81920 MiB
MulTaBench at c955466 (release-day master)
secrets loaded: ['HF_TOKEN', 'TABPFN_TOKEN', 'WANDB_API_KEY', 'WANDB_ENTITY']


In [ ]:
%%writefile /content/register_vaers.py
"""Register the VAERS dataset inside a MulTaBench checkout (release-day master c955466).

Adds exactly three things and changes nothing else:
  1. an entry in UrlDatasetID (multabench/datasets/all_datasets.py)
  2. a local-file mapping in get_csv_local_path (multabench/datasets/downloading.py)
  3. a curation recipe multabench/datasets/annotated/<DATASET_NAME>.py
and copies the CSV to multabench/datasets/url_datasets/.
Idempotent: running it twice does not duplicate anything.
Usage: python register_vaers.py <repo_dir> <csv_path> <expected_sha256>
"""
import hashlib
import shutil
import sys
from pathlib import Path

DATASET_NAME = "BIN_TEXT_VAERS_SERIOUS_V11B"
CSV_NAME = "vaers_v11b.csv"

RECIPE = f'''from multabench.datasets.curation_objects import CuratedFeature, CuratedTarget
from multabench.datasets.objects import SupervisedTask, FeatureType
from multabench.datasets.utils import load_csv

\'\'\'
Dataset Name: {DATASET_NAME}
====
Source: VAERS (Vaccine Adverse Event Reporting System), public CDC/FDA data, report years 1990-2024.
Target: serious = 1 if any of DIED, L_THREAT, HOSPITAL, X_STAY, DISABLE, BIRTH_DEFECT == 'Y'.
Text: patient_background (history/allergies/medications/illness at vaccination) and
      narrative_onset (opening of the symptom narrative, with outcome and care-process terms removed).
Structured: age, sex, state, onset delay, vaccine type/manufacturer/dose/route/site, number of vaccines,
            administering facility type, report year.
report_id is an audit column and is dropped.
\'\'\'

CONTEXT = "VAERS vaccine adverse event reports: predict whether the report describes a serious outcome"
TARGET = CuratedTarget(raw_name="serious", task_type=SupervisedTask.BINARY)
COLS_TO_DROP = ["report_id"]
FEATURES = [CuratedFeature(raw_name=c, feat_type=FeatureType.NUMERIC)
            for c in ["age_years", "onset_days", "n_vaccines", "report_year"]]


def load_df(dir_path):
    return load_csv(dir_path, "{CSV_NAME}")


LOADING_FUNC = load_df
'''


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def patch_once(path, anchor, insert, marker):
    text = path.read_text()
    if marker in text:
        return "already"
    assert text.count(anchor) == 1, f"anchor not found exactly once in {path}"
    path.write_text(text.replace(anchor, anchor + insert, 1))
    return "patched"


def main(repo, csv, expected):
    repo, csv = Path(repo), Path(csv)
    got = sha256(csv)
    print("csv sha256:", got)
    assert got == expected, "CSV hash does not match the frozen dataset -- STOP"
    ds = repo / "multabench" / "datasets"
    shutil.copy(csv, ds / "url_datasets" / CSV_NAME)
    print("enum:", patch_once(ds / "all_datasets.py", "class UrlDatasetID(Enum):\n",
                              f'    {DATASET_NAME} = "local://{CSV_NAME}"\n', DATASET_NAME))
    print("path map:", patch_once(ds / "downloading.py", "    dataset2local = {\n",
                                  f'        UrlDatasetID.{DATASET_NAME}: "{CSV_NAME}",\n', DATASET_NAME))
    (ds / "annotated" / f"{DATASET_NAME}.py").write_text(RECIPE)
    print("recipe: written")


if __name__ == "__main__":
    main(*sys.argv[1:4])

Overwriting /content/register_vaers.py


In [ ]:
# ===== 2. Register the dataset inside MulTaBench (hash-checked) =====
!python /content/register_vaers.py {REPO} {CSV_PATH} {CSV_SHA256}
!git -C {REPO} status --short

csv sha256: 8e59f09b19d0f7f174b38413ffe48052b005491fa1464877764d405911e16690
enum: patched
path map: patched
recipe: written
 M multabench/datasets/all_datasets.py
 M multabench/datasets/downloading.py
?? multabench/datasets/annotated/BIN_TEXT_VAERS_SERIOUS_V11B.py
?? multabench/datasets/url_datasets/vaers_v11b.csv


In [ ]:
# ===== 3. Install the official environment (Python 3.11 + the repo's pinned requirements.txt) =====
import os
os.environ["MPLBACKEND"] = "Agg"   # Colab's inline plot backend does not exist inside the venv
if not os.path.exists(f"{REPO}/.venv/INSTALLED_OK"):
    !pip -q install uv
    if not os.path.exists(f"{REPO}/.venv/bin/python"):
        !cd {REPO} && uv venv -q -p 3.11 .venv
    !cd {REPO} && UV_HTTP_TIMEOUT=300 uv pip install -q --python .venv/bin/python -r requirements.txt
    !{REPO}/.venv/bin/python -c "import torch, tabstar, lightgbm, catboost, tabpfn, wandb; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())" && touch {REPO}/.venv/INSTALLED_OK
assert os.path.exists(f"{REPO}/.venv/INSTALLED_OK"), "install failed - send Claude the output above"
print("environment ready")

environment ready


In [ ]:
# ===== 4. Quick check: the official loader reads the dataset =====
check = '''
from multabench.datasets.downloading import download_dataset
from multabench.datasets.utils import dataset_from_name
from multabench.datasets.multimodal import MultimodalState
d = dataset_from_name("%s")
ds = download_dataset(d)
print("LOAD OK", ds.task_type.name, ds.x.shape, "positives:", int(ds.y.astype(int).sum()))
for st in [MultimodalState.NO_TEXT, MultimodalState.TEXT_ONLY, MultimodalState.ALL]:
    print(" ", st.name, list(download_dataset(d, multimodal_state=st).x.columns))
''' % DATASET
open("/content/check_load.py", "w").write(check)
!cd {REPO} && PYTHONPATH={REPO} .venv/bin/python /content/check_load.py 2>&1 | grep -v -i warning

💾 Downloading URL dataset BIN_TEXT_VAERS_SERIOUS_V11B
LOAD OK BINARY (18859, 14) positives: 4900
💾 Downloading URL dataset BIN_TEXT_VAERS_SERIOUS_V11B
  NO_TEXT ['age_years', 'sex', 'state', 'onset_days', 'vax_type', 'vax_manu', 'vax_dose_series', 'vax_route', 'vax_site', 'n_vaccines', 'admin_by', 'report_year']
💾 Downloading URL dataset BIN_TEXT_VAERS_SERIOUS_V11B
  TEXT_ONLY ['patient_background', 'narrative_onset']
💾 Downloading URL dataset BIN_TEXT_VAERS_SERIOUS_V11B
  ALL ['age_years', 'sex', 'state', 'onset_days', 'vax_type', 'vax_manu', 'vax_dose_series', 'vax_route', 'vax_site', 'n_vaccines', 'admin_by', 'report_year', 'patient_background', 'narrative_onset']


In [ ]:
# ===== 5. Run everything: 5 learners x 4 conditions x 5 folds (resumable) =====
import csv, os, re, subprocess, time
RESULTS = f"{OUT_DIR}/results.csv"
FIELDS = ["model", "state", "fold", "status", "test_score", "runtime_s", "gpu", "commit", "finished_at"]

def done_keys():
    if not os.path.exists(RESULTS):
        return set()
    with open(RESULTS) as f:
        return {(r["model"], r["state"], int(r["fold"])) for r in csv.DictReader(f) if r["status"] == "ok"}

def append_row(row):
    new = not os.path.exists(RESULTS)
    with open(RESULTS, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        if new:
            w.writeheader()
        w.writerow(row)

env = dict(os.environ, **SECRETS, PYTHONPATH=REPO, TOKENIZERS_PARALLELISM="false", WANDB_SILENT="true")
plan = [(m, s, f) for f in FOLDS for m in MODELS for s in STATES]   # fold by fold
already = done_keys()
todo = [p for p in plan if p not in already]
print(f"{len(already)} runs already done, {len(todo)} to go")

t_start, n_new = time.time(), 0
for i, (m, s, f) in enumerate(todo, 1):
    log = f"{OUT_DIR}/logs/{m}_{s}_fold{f}.log"
    t0 = time.time()
    with open(log, "w") as lf:
        p = subprocess.run([f"{REPO}/.venv/bin/python", "benchmark.py", "--model", m,
                            "--dataset_name", DATASET, "--fold", str(f),
                            "--multimodal_state", s, "--project", WANDB_PROJECT],
                           cwd=REPO, env=env, stdout=lf, stderr=subprocess.STDOUT)
    secs = time.time() - t0
    text = open(log, errors="replace").read()
    hit = re.search(r"'test_score': ([-0-9.eE+]+)", text)
    ok = p.returncode == 0 and hit is not None
    append_row({"model": m, "state": s, "fold": f, "status": "ok" if ok else "failed",
                "test_score": hit.group(1) if hit else "", "runtime_s": round(secs, 1),
                "gpu": gpu, "commit": COMMIT[:7], "finished_at": time.strftime("%Y-%m-%d %H:%M:%S")})
    n_new += 1
    eta_h = (time.time() - t_start) / n_new * (len(todo) - i) / 3600
    print(f"[{len(already)+i}/{len(plan)}] {m:10s} {s:9s} fold {f}: "
          f"{(hit.group(1)[:8] if hit else 'FAILED - see ' + log)}  ({secs/60:.1f} min)  ~{eta_h:.1f} h left")
print("ALL RUNS FINISHED" if len(done_keys()) == len(plan) else "Some runs missing/failed - run this cell again")

17 runs already done, 83 to go
[18/100] light      ft        fold 0: 0.849940  (14.7 min)  ~20.1 h left
[19/100] cat        ft        fold 0: 0.859654  (14.9 min)  ~19.9 h left
[20/100] tabm       ft        fold 0: 0.859294  (14.7 min)  ~19.7 h left
[21/100] tabpfnv2   ft        fold 0: 0.859812  (14.5 min)  ~19.3 h left
[22/100] tabpfnv2p5 ft        fold 0: 0.867358  (14.4 min)  ~19.0 h left
[23/100] light      all       fold 1: 0.799814  (2.1 min)  ~16.1 h left
[24/100] light      ft        fold 1: 0.824497  (11.3 min)  ~15.7 h left
[25/100] cat        no_text   fold 1: 0.734452  (0.5 min)  ~13.6 h left
[26/100] cat        text_only fold 1: 0.770972  (2.2 min)  ~12.2 h left
[27/100] cat        all       fold 1: 0.816130  (2.4 min)  ~11.2 h left
[28/100] cat        ft        fold 1: 0.823426  (11.5 min)  ~11.3 h left
[29/100] tabm       no_text   fold 1: 0.728613  (0.4 min)  ~10.2 h left
[30/100] tabm       text_only fold 1: 0.783658  (2.2 min)  ~9.5 h left
[31/100] tabm       all    

In [10]:
# ===== 6. Summary: the two MulTaBench curation checks (raw scores, 3 of 5 learners per check) =====
import pandas as pd
r = pd.read_csv(RESULTS)
r = r[r.status == "ok"].drop_duplicates(["model", "state", "fold"], keep="last")
counts = r.groupby(["model", "state"]).size().unstack()
print("runs per model/condition (need 5 each):"); print(counts.to_string(), "\n")

means = r.groupby(["model", "state"])["test_score"].mean().unstack()[STATES]
means["joint_gain"] = means["all"] - means[["no_text", "text_only"]].max(axis=1)
means["tar_gain"]   = means["ft"] - means["all"]
means["joint_pass"] = means["joint_gain"] > 0
means["tar_pass"]   = means["tar_gain"] > 0
pd.set_option("display.float_format", "{:.4f}".format)
print(means.to_string(), "\n")
complete = bool((counts.reindex(index=MODELS, columns=STATES) == 5).all().all())
print(f"Joint Signal passes: {int(means.joint_pass.sum())}/5   ->", "PASS" if means.joint_pass.sum() >= 3 else "FAIL")
print(f"TAR Gain passes:     {int(means.tar_pass.sum())}/5   ->", "PASS" if means.tar_pass.sum() >= 3 else "FAIL")
print(f"median joint gain {means.joint_gain.median():+.4f} | median TAR gain {means.tar_gain.median():+.4f}")
print("ALL 100 RUNS COMPLETE" if complete else "NOT COMPLETE YET - numbers above are partial")

# reference only: the paper's stricter rule (means rounded to 3 decimals, both gains > 0.001)
rr = r.groupby(["model", "state"])["test_score"].mean().round(3).unstack()
paper = ((rr["all"] - rr[["no_text", "text_only"]].max(axis=1) > 0.001) & (rr["ft"] - rr["all"] > 0.001))
print("paper-rule passes (reference):", int(paper.sum()), "/ 5")

per_fold = r.pivot_table(index=["model", "fold"], columns="state", values="test_score")[STATES]
per_fold["joint_gain"] = per_fold["all"] - per_fold[["no_text", "text_only"]].max(axis=1)
per_fold["tar_gain"] = per_fold["ft"] - per_fold["all"]
means.to_csv(f"{OUT_DIR}/summary.csv"); per_fold.to_csv(f"{OUT_DIR}/per_fold.csv")
!cd {REPO} && git diff > {OUT_DIR}/registration.diff
print("saved summary.csv, per_fold.csv, registration.diff to", OUT_DIR)

runs per model/condition (need 5 each):
state       all  ft  no_text  text_only
model                                  
cat           5   5        5          5
light         5   5        5          5
tabm          5   5        5          5
tabpfnv2      5   5        5          5
tabpfnv2p5    5   5        5          5 

state       no_text  text_only    all     ft  joint_gain  tar_gain  joint_pass  tar_pass
model                                                                                   
cat          0.7393     0.7871 0.8231 0.8432      0.0360    0.0201        True      True
light        0.7214     0.7777 0.8090 0.8394      0.0312    0.0304        True      True
tabm         0.7254     0.7989 0.8319 0.8474      0.0330    0.0155        True      True
tabpfnv2     0.7354     0.7946 0.8212 0.8428      0.0266    0.0215        True      True
tabpfnv2p5   0.7396     0.8051 0.8345 0.8507      0.0294    0.0162        True      True 

Joint Signal passes: 5/5   -> PASS
TAR Gain passes:  